### Here is the implemetation of "Fast Semi-Supervised Node Embedding Learning via Structural and Label-Aware Optimization"



In [ ]:
import pandas as pd 
import numpy as np 
import os
import networkx as nx
import random
import torch
from torch_geometric.utils import to_dense_adj







### reading the dataset :



In [181]:

data_dir = os.path.expanduser("cora")

edgelist = pd.read_csv(os.path.join(data_dir, "cora.cites"), sep='\t', header=None, names=["target", "source"])
edgelist["label"] = "cites"

In [182]:
edgelist.head(5)

,target,source,label
0,35,1033,cites
1,35,103482,cites
2,35,103515,cites
3,35,1050679,cites
4,35,1103960,cites


In [183]:
Gnx = nx.from_pandas_edgelist(edgelist, edge_attr="label")
nx.set_node_attributes(Gnx, "paper", "label")

In [184]:
Gnx.nodes[1103985]

{'label': 'paper'}

In [185]:
feature_names = ["w_{}".format(ii) for ii in range(1433)]
column_names =  feature_names + ["subject"]
node_data = pd.read_csv(os.path.join(data_dir, "cora.content"), sep='\t', header=None, names=column_names)

In [186]:
node_data.head(5)

,w_0,w_1,w_2,w_3,w_4,w_5,w_6,w_7,w_8,w_9,...,w_1424,w_1425,w_1426,w_1427,w_1428,w_1429,w_1430,w_1431,w_1432,subject
31336,0,0,0,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,Neural_Networks
1061127,0,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,Rule_Learning
1106406,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Reinforcement_Learning
13195,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Reinforcement_Learning
37879,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Probabilistic_Methods


In [187]:
node_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2708 entries, 31336 to 24043
Columns: 1434 entries, w_0 to subject
dtypes: int64(1433), object(1)
memory usage: 29.6+ MB


In [188]:
node_data.shape

(2708, 1434)

In [189]:
set(node_data["subject"])


{'Case_Based',
 'Genetic_Algorithms',
 'Neural_Networks',
 'Probabilistic_Methods',
 'Reinforcement_Learning',
 'Rule_Learning',
 'Theory'}

In [190]:
node_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2708 entries, 31336 to 24043
Columns: 1434 entries, w_0 to subject
dtypes: int64(1433), object(1)
memory usage: 29.6+ MB


## Getting Started:

In [191]:
from torch_geometric.datasets import Planetoid
dataset = Planetoid(root='Cora', name='Cora')

data = dataset[0]
print(f'Dataset: {dataset}:')
print('======================')
print(f'Number of graphs: {len(dataset)}')
print(f'Number of features: {dataset.num_features}')
print(f'Number of classes: {dataset.num_classes}')

print(f'Number of nodes: {data.num_nodes}')
print(f'Number of edges: {data.num_edges}')
print(f'Average node degree: {data.num_edges / data.num_nodes:.2f}')
print(f'Number of training nodes: {data.train_mask.sum()}')
print(f'Training node label rate: {int(data.train_mask.sum()) / data.num_nodes:.2f}')
print(f'Contains isolated nodes: {data.contains_isolated_nodes()}')
print(f'Contains self-loops: {data.contains_self_loops()}')
print(f'Is undirected: {data.is_undirected()}')


Dataset: Cora():
Number of graphs: 1
Number of features: 1433
Number of classes: 7
Number of nodes: 2708
Number of edges: 10556
Average node degree: 3.90
Number of training nodes: 140
Training node label rate: 0.05
Contains isolated nodes: False
Contains self-loops: False
Is undirected: True


C:\Users\mehul\AppData\Local\Temp\ipykernel_19796\1433470767.py:16: UserWarning: 'contains_isolated_nodes' is deprecated, use 'has_isolated_nodes' instead
  print(f'Contains isolated nodes: {data.contains_isolated_nodes()}')
C:\Users\mehul\AppData\Local\Temp\ipykernel_19796\1433470767.py:17: UserWarning: 'contains_self_loops' is deprecated, use 'has_self_loops' instead
  print(f'Contains self-loops: {data.contains_self_loops()}')


### Init.

In [192]:
device = "cpu"

In [193]:
original_embed = data.x

# Adjacency matrix
A = to_dense_adj(data.edge_index)[0]

# Labels for each node:
labels = data.y

# Verification of dimensions
print(f'Original embedding shape: {original_embed.shape}')     # [2708, 1433]
print(f'Adjacency Matrix shape: {A.shape}') # [2708, 2708]
print(f'Labels shape: {labels.shape}')               # [2708]

Original embedding shape: torch.Size([2708, 1433])
Adjacency Matrix shape: torch.Size([2708, 2708])
Labels shape: torch.Size([2708])


In [194]:
# initial random feature ,atrix :

S = torch.rand(2708, 145, device=device)  

print(f'Random embedding shape: {S.shape}')

Random embedding shape: torch.Size([2708, 145])


In [195]:
# Qr decomposition of the random feature matrix:

Q, R = torch.linalg.qr(S)
S_ortho = Q
print(f'Orthogonalized embedding shape: {S_ortho.shape}')

Orthogonalized embedding shape: torch.Size([2708, 145])


In [196]:
S = S_ortho

In [197]:
A = A.to(device)
S = S.to(device)
labels = labels.to(device)

### Masking the labels:

- masking rate : 0.5 (MCAR) 


In [198]:
labels

tensor([3, 4, 4,  ..., 3, 3, 3])

In [199]:
# masking 50 poercent of the lables at random:
num_nodes = labels.shape[0]
num_mask = int(0.5 * num_nodes)
mask_indices = torch.randperm(num_nodes)[:num_mask]
labels_masked = labels.clone()
labels_masked[mask_indices] = -1

In [200]:
A

tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 1.,  ..., 0., 0., 0.],
        [0., 1., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 1.],
        [0., 0., 0.,  ..., 0., 1., 0.]])

### Biased Random Walk :
(Hyperparams for Cora Dataset)

- r = 20
- L = 4 (l)
- L'= 1 (L)

In [201]:
labels_masked = labels_masked.to(device)

so here I need to map the nodes from labels_masked, A and S, to calcualte the 

In [202]:
labels_list = labels_masked

In [203]:
num_nodes = labels_list.shape[0]
adj_list = [[] for i in range(num_nodes)]
edges = A.nonzero(as_tuple=False)
edges_list = edges.tolist()
for src, dst in edges_list:
        adj_list[src].append(dst)


In [204]:
def random_walk(labels_list, adj_list, r, l, L, device = device):
 
    
    W_f = {}  # labelled nodes list for all the unlabeled nodes
    
    for node_id, label_value in enumerate(labels_list):
        
        if label_value == -1:
            W = []
            
            for walk in range(r):
                current_node = node_id
                labeled_count = 0
                for step in range(l):
                    neighbors = adj_list[current_node]
                    if not neighbors:
                        break
                    next_node = random.choice(neighbors) # randomly pick the neighbor 
                    if labels_list[next_node] != -1:
                        if labeled_count < L:
                            W.append(next_node)
                            labeled_count += 1
                        else:
                            break
                            # moving to the next node:
                    current_node = next_node
            
            W_f[node_id] = W
            
    return W_f



In [205]:
W_f = random_walk(labels_list, adj_list, r=20, l=4, L=1)


here is the list of labelled neighbors obtained for the first masked node:

In [206]:
len(W_f) # lists for the masked nodes
 

1354

In [207]:
import torch

def modularity(A, edgelist, S, device = device):

    m = len(edgelist)
    d = A.sum(dim=1, keepdim=True) 
    TS = S.sum(dim=0, keepdim=True)
    AS = A @ S  
    degree_correction = d * TS
    factor = 1.0 / (2 * m)
    Q_prop = factor * (AS - factor * degree_correction)
    
    return Q_prop




In [208]:

def sup_reg(S, labels_list, device = device):

    
    grad_sup = torch.zeros_like(S, device=device)
    
    label_set = set(labels_list)
    
    for current_class in label_set:
        if current_class == -1:
            continue

        node_indices = [i for i, label in enumerate(labels_list) if label == current_class]
        class_embeddings = S[node_indices]
        mu_c = class_embeddings.mean(dim=0)
        grad_sup[node_indices] = class_embeddings - mu_c
        
    return grad_sup

In [209]:
W_f.items()

dict_items([(5, [1765, 1358]), (8, [269, 1996, 1996, 269, 281, 1996, 281, 281, 269, 269, 281, 281, 1996, 1996, 269, 269, 1996, 1996, 281, 1996]), (9, []), (11, [1839, 1839, 2384, 1839, 1842, 1839, 1839, 1839, 1839, 1839, 1839, 1839, 1013, 1839, 1839]), (15, [1271, 1093, 2367, 1271, 1093, 1093, 1271, 1271, 2367, 1271, 1271, 1271, 2367, 2367, 2367, 2367, 2367, 1271, 1271]), (16, [2444, 364, 364, 2444, 2444, 364, 364, 364, 364, 2444, 364, 2444, 364, 2444, 364, 2444, 364, 1358]), (18, [1448, 1623, 589, 2145, 1772, 1623, 1623, 505, 308, 1772, 2145, 699, 109, 2145, 910, 2145, 2145]), (20, [1825, 2374, 2374, 1072, 1404, 2374, 1825, 2269, 2269, 2269, 1404, 400, 2269, 1072, 1072, 1072, 2374, 1404]), (24, [2117, 17, 2142, 297, 1986, 1986, 17, 343, 467, 143, 2706]), (25, [1301, 1301, 1301, 2317, 2317, 2317, 2011, 1301, 1301, 1301, 2011, 2011, 2317, 1301, 2011, 1301, 1301, 2011, 1301, 2011]), (26, [123, 99, 123, 99, 2455, 2455, 2455, 123, 123, 2455, 99, 2455, 99, 2455, 123, 99, 99]), (27, [1809, 5

In [210]:
import torch

def attention_wts(W_f, S, device = device):

    attention_wts = {}
    for node, labelled_neigh in W_f.items():
        if not labelled_neigh:
            attention_wts[node] = torch.tensor([], device=device)  
            continue

        neighbors_idx = torch.tensor(labelled_neigh, device=device)
        s_i = S[node]
        
        s_neighbors = S[neighbors_idx]
        
        sim = torch.matmul(s_neighbors, s_i)
        
        # Apply softmax across the sim to get the final weights
        # This executes the exp(s_ij) / sum(exp(s_ik)) formula exactly (Equation 7)
        w_ij = torch.softmax(sim, dim=0)
        
        attention_wts[node] = w_ij
        
    return attention_wts

In [211]:
# saving the index of the masked nodes to be considered for semi supervised loss :

masked = []

for idx, nodes in enumerate(labels_list):
    if nodes == -1:
        masked.append(idx)

In [212]:
masked

[5,
 8,
 9,
 11,
 15,
 16,
 18,
 20,
 24,
 25,
 26,
 27,
 28,
 29,
 30,
 31,
 32,
 35,
 37,
 38,
 43,
 44,
 47,
 48,
 49,
 50,
 54,
 55,
 56,
 59,
 60,
 62,
 63,
 66,
 67,
 69,
 72,
 73,
 74,
 76,
 78,
 80,
 81,
 87,
 88,
 90,
 92,
 93,
 95,
 96,
 98,
 100,
 101,
 106,
 108,
 110,
 116,
 117,
 118,
 119,
 122,
 125,
 127,
 128,
 129,
 130,
 131,
 132,
 133,
 136,
 138,
 139,
 142,
 144,
 146,
 147,
 148,
 149,
 150,
 151,
 152,
 153,
 156,
 159,
 160,
 162,
 163,
 164,
 166,
 168,
 169,
 172,
 173,
 174,
 180,
 181,
 182,
 184,
 187,
 188,
 189,
 190,
 192,
 193,
 194,
 195,
 196,
 197,
 199,
 201,
 203,
 204,
 205,
 206,
 210,
 211,
 212,
 213,
 225,
 226,
 227,
 228,
 229,
 232,
 233,
 234,
 237,
 240,
 243,
 247,
 249,
 250,
 252,
 253,
 254,
 257,
 258,
 259,
 260,
 263,
 264,
 265,
 266,
 268,
 270,
 272,
 279,
 280,
 287,
 288,
 290,
 291,
 292,
 294,
 296,
 299,
 300,
 301,
 303,
 304,
 306,
 312,
 314,
 317,
 318,
 320,
 323,
 326,
 329,
 330,
 331,
 334,
 335,
 337,
 338,
 340

In [213]:


def semi_sup(S, masked, W_f, attention_wts, device = device):

    grad_semi = torch.zeros_like(S, device=device)
    
    for i in masked:
        neighbors_list = W_f[i]
        
        if not neighbors_list:
            continue
            
        w_ij = attention_wts[i]
        
        neighbors_idx = torch.tensor(neighbors_list, dtype=torch.long, device=S.device)
        S_j = S[neighbors_idx]
        
        weighted_sum = (w_ij.unsqueeze(1) * S_j).sum(dim=0)
        

        grad_semi[i] = S[i] - weighted_sum
        
    return grad_semi

In [214]:
# optimization loop :
lr = 0.31
lambda_sup = 0.6
lambda_semi = 1.9


for i in range(200):
    
    wij = attention_wts(W_f, S)
    
    Q_semi = semi_sup(S, masked, W_f, wij)
    Q_prop = modularity(A, edgelist, S)
    Q_sup = sup_reg(S, labels_list)
    
    S = S - lr*(Q_prop - lambda_sup*Q_sup - lambda_semi*Q_semi)
    
    Q, R = torch.linalg.qr(S)
    S = Q
    print(f"Iteration {i+1} completed.")


Iteration 1 completed.
Iteration 2 completed.
Iteration 3 completed.
Iteration 4 completed.
Iteration 5 completed.
Iteration 6 completed.
Iteration 7 completed.
Iteration 8 completed.
Iteration 9 completed.
Iteration 10 completed.
Iteration 11 completed.
Iteration 12 completed.
Iteration 13 completed.
Iteration 14 completed.
Iteration 15 completed.
Iteration 16 completed.
Iteration 17 completed.
Iteration 18 completed.
Iteration 19 completed.
Iteration 20 completed.
Iteration 21 completed.
Iteration 22 completed.
Iteration 23 completed.
Iteration 24 completed.
Iteration 25 completed.
Iteration 26 completed.
Iteration 27 completed.
Iteration 28 completed.
Iteration 29 completed.
Iteration 30 completed.
Iteration 31 completed.
Iteration 32 completed.
Iteration 33 completed.
Iteration 34 completed.
Iteration 35 completed.
Iteration 36 completed.
Iteration 37 completed.
Iteration 38 completed.
Iteration 39 completed.
Iteration 40 completed.
Iteration 41 completed.
Iteration 42 completed.
I

In [215]:
S
torch.save(S, 'S_cora.pt')  # Saves tensor to 'tensor.pt' in the current directory

### Training MLP for the node classification: S_generated

In [ ]:
# train test split :


S_generated = torch.load('S_cora.pt')  

S_train = S_generated[labels_masked != -1]
S_test = S_generated[labels_masked == -1]




C:\Users\mehul\AppData\Local\Temp\ipykernel_19796\2835061828.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  S_generated = torch.load('S_cora.pt')


In [217]:
S_train.shape

torch.Size([1354, 145])

In [218]:
S_test.shape

torch.Size([1354, 145])

In [221]:
labels_train = labels_masked[labels_masked != -1]
labels_train.shape


torch.Size([1354])

In [226]:
import torch
import torch.nn as nn
import torch.optim as optim

class smol_mp(nn.Module):
    def __init__(self, input_dim, layers=[256, 128, 64, 32], num_classes=7):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, layers[0]),
            nn.ReLU(),
            nn.Linear(layers[0], layers[1]),
            nn.ReLU(),
            nn.Linear(layers[1], layers[2]),
            nn.ReLU(),
            nn.Linear(layers[2], layers[3]),
            nn.ReLU(),
            nn.Linear(layers[3], num_classes)
        )

    def forward(self, x):
        return self.net(x)
    

smol1 = smol_mp(input_dim=145) 

labels_train = labels_train.clone().detach().long()# CrossEntropy expects Long type
S = S_generated

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(smol1.parameters(), lr= 0.01)

epochs = 10000
smol1.train() 

for epoch in range(epochs):
    optimizer.zero_grad()                     
    
    logits_train = smol1(S_train)              
    loss = criterion(logits_train, labels_train) 
    
    loss.backward()                            
    optimizer.step()                           
    
    # Optional: Print progress
    if (epoch + 1) % 2 == 0:
        print(f"Epoch {epoch + 1}/{epochs} | Loss: {loss.item():.4f}")








Epoch 2/10000 | Loss: 1.9222
Epoch 4/10000 | Loss: 1.8679
Epoch 6/10000 | Loss: 1.8501
Epoch 8/10000 | Loss: 1.8193
Epoch 10/10000 | Loss: 1.8302
Epoch 12/10000 | Loss: 1.8254
Epoch 14/10000 | Loss: 1.8173
Epoch 16/10000 | Loss: 1.8196
Epoch 18/10000 | Loss: 1.8167
Epoch 20/10000 | Loss: 1.8168
Epoch 22/10000 | Loss: 1.8173
Epoch 24/10000 | Loss: 1.8159
Epoch 26/10000 | Loss: 1.8158
Epoch 28/10000 | Loss: 1.8162
Epoch 30/10000 | Loss: 1.8155
Epoch 32/10000 | Loss: 1.8155
Epoch 34/10000 | Loss: 1.8155
Epoch 36/10000 | Loss: 1.8152
Epoch 38/10000 | Loss: 1.8153
Epoch 40/10000 | Loss: 1.8154
Epoch 42/10000 | Loss: 1.8152
Epoch 44/10000 | Loss: 1.8152
Epoch 46/10000 | Loss: 1.8152
Epoch 48/10000 | Loss: 1.8151
Epoch 50/10000 | Loss: 1.8152
Epoch 52/10000 | Loss: 1.8152
Epoch 54/10000 | Loss: 1.8151
Epoch 56/10000 | Loss: 1.8151
Epoch 58/10000 | Loss: 1.8151
Epoch 60/10000 | Loss: 1.8151
Epoch 62/10000 | Loss: 1.8151
Epoch 64/10000 | Loss: 1.8151
Epoch 66/10000 | Loss: 1.8151
Epoch 68/10000

KeyboardInterrupt: 